### MIMII Audio to Mel Spectrogram

Place this script anywhere. Define root as your MIMII dataset folder.

your directory tree should looks like:

```
root
  ├──fan
  │   ├──id_00
  │   │    ├──normal
  │   │    └──abnormal
  │   ├──id_02
  │   │    ├──normal
  │   │    └──abnormal
  │   ├──id_04
  │   │    ├──normal
  │   │    └──abnormal
  │   └──id_06
  │        ├──normal
  │        └──abnormal
  │
  ├──valve
      .
      .
      .
 (same for all 4 machine types)
 ```

In [13]:
import glob, os
import librosa 
import numpy as np
import cv2
from scipy.io import wavfile
from matplotlib import pyplot as plt
import wave

In [14]:
# define your own dataset root

root = '../tmp/dcase-2020/'
output_root = '../tmp/dcase-2020-spectrogram/'

In [15]:
# tuple of (input path, output path)
cats = [('data_fan/fan', 'fan'),
		('data_valve/valve', 'valve'),
		('data_pump/pump', 'pump'),
		('data_slider/slider', 'slider')]
# in every category, there are these subcategories
subs = ['test', 'train']
# labels of (input, output)
labels = [('normal_', 'normal'), ('anomaly_', 'abnormal')]

In [16]:
def get_paths(cat, sub, label, suffix='out'):
    path = root + cat[0] + '/' + sub + '/' + label[0]
    targetpath = output_root + cat[1] + '/' + sub + '_' + suffix + '/' + label[1] + '/'
    
    print(path)
    print(targetpath)
    
    # if not os.path.exists(path):
    #     raise FileNotFoundError(f"Path not found: {path}")
    if not os.path.exists(targetpath):
        print("new target path created.")
        os.makedirs(targetpath)

    return path, targetpath

In [17]:
def scale_minmax(X, smin=0.0, smax=1.0):
    X_std = (X - X.min()) / (X.max() - X.min())
    X_scaled = X_std * (smax - smin) + smin
    return X_scaled

def save_mel_img(fname, oname, targetpath, hop_length=512, n_mels=128, fmax=8000):
    y, sr = librosa.load(fname, sr=None)
    S = librosa.feature.melspectrogram(y=y, 
                                       sr=sr, 
                                       n_mels=n_mels, 
                                       n_fft=hop_length*2, 
                                       hop_length=hop_length, 
                                       fmax=fmax,
                                      )
    S = librosa.power_to_db(S, ref=np.max)
    img = scale_minmax(S, 0, 255).astype(np.uint8)
    img = np.flip(img, axis=0)
    
    # plt.figure(figsize=(4,2))
    # plt.imshow(img)
    # plt.show()
    
    savename = targetpath + oname + '.png'
    print(savename, S.shape, y.shape[0])
    cv2.imwrite(savename, img)

In [18]:
%%time
import re


for label in labels:
    pattern = re.compile(rf"^{label[0]}(.*)\.wav$")
    for cat in cats:
        for sub in subs:
            path, targetpath = get_paths(cat, sub, label)
            fnames = glob.glob(path+'*.wav')
            print("folder size:", len(fnames))

            for fname in fnames:
                m = pattern.match(os.path.basename(fname))
                if m:
                    save_mel_img(fname, m.group(1), targetpath)

../tmp/dcase-2020/data_fan/fan/test/normal_
../tmp/dcase-2020-spectrogram/fan/test_out/normal/
new target path created.
folder size: 400
../tmp/dcase-2020-spectrogram/fan/test_out/normal/id_00_00000099.png (128, 313) 160000
../tmp/dcase-2020-spectrogram/fan/test_out/normal/id_02_00000098.png (128, 313) 160000
../tmp/dcase-2020-spectrogram/fan/test_out/normal/id_00_00000000.png (128, 313) 160000
../tmp/dcase-2020-spectrogram/fan/test_out/normal/id_00_00000001.png (128, 313) 160000
../tmp/dcase-2020-spectrogram/fan/test_out/normal/id_00_00000002.png (128, 313) 160000
../tmp/dcase-2020-spectrogram/fan/test_out/normal/id_00_00000003.png (128, 313) 160000
../tmp/dcase-2020-spectrogram/fan/test_out/normal/id_00_00000004.png (128, 313) 160000
../tmp/dcase-2020-spectrogram/fan/test_out/normal/id_00_00000005.png (128, 313) 160000
../tmp/dcase-2020-spectrogram/fan/test_out/normal/id_00_00000006.png (128, 313) 160000
../tmp/dcase-2020-spectrogram/fan/test_out/normal/id_00_00000007.png (128, 313) 